# Etapa 6 - Projecao de compras para o proximo periodo

Notebook executavel da Etapa 6. A logica canonica fica em `etapa6_projecao_compras.py`; este notebook executa o script e registra uma leitura dos outputs auditaveis gerados em `outputs/etapa6/`.

A etapa transforma cobertura, demanda historica, custo valido e sinais de margem em um plano conservador de compra para 90 dias, separando `REDE_FISICA_SEM_LOJA93` e `LOJA_93_ATACADO_B2B`, com reconciliacao em `REDE_COMPLETA`.

## Execucao canonica

Toda metrica importante e persistida em CSV ou Markdown. O notebook deve ser lido como registro de execucao e leitura dos arquivos finais, nao como fonte paralela de regra de negocio.

In [1]:
from pathlib import Path
import runpy

candidatos = [Path("etapa6_projecao_compras.py"), Path("notebooks/etapa6_projecao_compras.py")]
script = next(p for p in candidatos if p.exists())
_ = runpy.run_path(str(script), run_name="__main__")

Carregando cobertura, vendas e sinais das etapas anteriores...


Calculando plano operacional de 90 dias...


Construindo autoaudit e validacoes...
Salvando arquivos auditaveis...



--- Destaques Etapa 6 ---
Pares com compra recomendada - rede completa: 20,357
Quantidade recomendada - rede fisica: 993,649 unid. armazenagem
Investimento conhecido - rede fisica: R$ 5.7M
Cobertura de custo dos pares recomendados - rede completa: 9.7%
Validacoes OK: 17/17

[OK] Arquivos salvos em outputs/etapa6/
  outputs\etapa6\autoaudit_etapa6.csv
  outputs\etapa6\documentacao_tecnica_etapa6.md
  outputs\etapa6\plano_compras_categorias_n1.csv
  outputs\etapa6\plano_compras_lojas.csv
  outputs\etapa6\plano_compras_sku_loja.csv
  outputs\etapa6\plano_compras_total_universo.csv
  outputs\etapa6\priorizacao_compras.csv
  outputs\etapa6\recomendacoes_melhoria.csv
  outputs\etapa6\resumo_etapa6.md
  outputs\etapa6\validacoes_etapa6.csv


## KPIs persistidos

Resumo lido diretamente de `outputs/etapa6/plano_compras_total_universo.csv`.

In [2]:
import pandas as pd

out = Path("outputs/etapa6") if Path("outputs/etapa6").exists() else Path("../outputs/etapa6")
total = pd.read_csv(out / "plano_compras_total_universo.csv", encoding="utf-8-sig")
total[[
    "UNIVERSO",
    "PARES_COM_COMPRA_RECOMENDADA",
    "QTD_RECOMENDADA_ARM",
    "INVESTIMENTO_ESTIMADO_COM_CUSTO",
    "COBERTURA_CUSTO_PARES_COMPRA_PCT",
]]

,UNIVERSO,PARES_COM_COMPRA_RECOMENDADA,QTD_RECOMENDADA_ARM,INVESTIMENTO_ESTIMADO_COM_CUSTO,COBERTURA_CUSTO_PARES_COMPRA_PCT
0,REDE_COMPLETA,20357,1019914.0,7.719604e+06,9.672349
1,REDE_FISICA_SEM_LOJA93,20061,993649.0,5.665536e+06,9.740292
2,LOJA_93_ATACADO_B2B,296,26265.0,2.054068e+06,5.067568


## Validacoes

Todas as reconciliacoes numericas precisam estar `OK` antes de usar a etapa.

In [3]:
validacoes = pd.read_csv(out / "validacoes_etapa6.csv", encoding="utf-8-sig")
print(f"Validacoes OK: {(validacoes['STATUS'] == 'OK').sum()}/{len(validacoes)}")
validacoes

Validacoes OK: 17/17


,VALIDACAO,OBSERVADO,ESPERADO,DIFERENCA,STATUS
0,Receita historica plano vs Etapa 3 - rede comp...,4.825157e+08,4.825157e+08,0.0,OK
1,Receita historica plano vs Etapa 3 - rede fisica,3.292348e+08,3.292348e+08,0.0,OK
2,Receita historica plano vs Etapa 3 - Loja 93,1.532810e+08,1.532810e+08,0.0,OK
3,REDE_COMPLETA = fisica + Loja 93 (PARES_LOJA_SKU),2.872100e+04,2.872100e+04,0.0,OK
4,REDE_COMPLETA = fisica + Loja 93 (RECEITA_HIST...,4.825157e+08,4.825157e+08,-0.0,OK
5,REDE_COMPLETA = fisica + Loja 93 (QTD_RECOMEND...,1.019914e+06,1.019914e+06,0.0,OK
6,Investimento conhecido: REDE_COMPLETA = fisica...,7.719604e+06,7.719604e+06,-0.0,OK
7,Categoria N1: receita total vs Etapa 4 - rede ...,0.000000e+00,0.000000e+00,0.0,OK
8,Toda compra recomendada possui demanda observada,1.000000e+00,1.000000e+00,0.0,OK
9,Toda compra recomendada esta em status elegivel,1.000000e+00,1.000000e+00,0.0,OK


## Autoaudit

Revisao critica dos principais riscos de interpretacao e falsos positivos.

In [4]:
pd.read_csv(out / "autoaudit_etapa6.csv", encoding="utf-8-sig")

,RISCO,COMO_PODERIA_ERRAR,CONTROLE_APLICADO,EVIDENCIA,RISCO_REMANESCENTE
0,Comprar para pares sem demanda observada,Tratar ausencia de venda como zero e ainda ass...,Quantidade recomendada so e calculada quando e...,0 compras recomendadas sem demanda observada.,A demanda historica pode estar subestimada se ...
1,Orcamento falso para itens sem custo,Multiplicar quantidade por zero ou por custo m...,Investimento fica nulo quando o SKU nao tem cu...,18388 pares recomendados sem custo e 1969 com ...,O plano operacional pode ser maior que o orcam...
2,Apagar demanda da Loja 93/B2B,Reusar a venda media da Etapa 2 (que excluiu a...,Demanda e status de cobertura da Etapa 6 sao r...,327 pares da Loja 93 tinham venda observada e ...,B2B pode ter pedidos grandes e intermitentes; ...
3,Interpretar estoque negativo como quantidade a...,Somar o negativo a demanda futura e inflar com...,"Estoque negativo vira zero utilizavel, nao uma...",18453 pares com estoque projetado negativo tra...,"Se o negativo refletir venda nao reposta, aind..."


## Arquivos gerados

- `outputs/etapa6/plano_compras_sku_loja.csv`
- `outputs/etapa6/plano_compras_total_universo.csv`
- `outputs/etapa6/plano_compras_categorias_n1.csv`
- `outputs/etapa6/plano_compras_lojas.csv`
- `outputs/etapa6/priorizacao_compras.csv`
- `outputs/etapa6/recomendacoes_melhoria.csv`
- `outputs/etapa6/validacoes_etapa6.csv`
- `outputs/etapa6/autoaudit_etapa6.csv`
- `outputs/etapa6/resumo_etapa6.md`
- `outputs/etapa6/documentacao_tecnica_etapa6.md`